In [33]:
import pandas as pd
import json 
import io
import zstandard as zstd
from tqdm import tqdm
import os

In [ ]:
filepath = "./reddit/subreddits/subreddits_2025-01.zst"
print(os.path.getsize(filepath) / 1e6, "MB")

2240.562367 MB


### Function for loading zst files

In [ ]:

#Bruger chunks så den ikke crasher pga memory overload
def load_zst_json(filepath, chunk_size=10000):
    chunks = []
    buffer = []


    with open(filepath, "rb") as f:
        dctx = zstd.ZstdDecompressor()
        with dctx.stream_reader(f) as reader:
            text_stream = io.TextIOWrapper(reader, encoding="utf-8")

            num_dataframes=0
            for i, line in tqdm(enumerate(text_stream), desc= "Amount of subreddits processed"):
                buffer.append(json.loads(line)) #Converts each json line to python dictionary

                if len(buffer) >= chunk_size:
                    chunks.append(pd.DataFrame(buffer))
                    buffer = []
                    #Tilføj denne linje hvis du kun vil have et subset ud. 100 svarer til 100000 subreddits
                    #if len(chunks) >= 100: return pd.concat(chunks, ignore_index=True)

                    # save file for each 1 million subreddits
                    if len(chunks) >= 100:
                        df=pd.concat(chunks,ignore_index=True)
                        df.to_csv(f"./data/subreddits/dataframe_{num_dataframes}.csv", index=False)
                        # reset chunks and buffer
                        chunks = []
                        buffer = []
                        num_dataframes+=1

            # remaining rows
            if buffer:
                chunks.append(pd.DataFrame(buffer))

    # save the remaining chunks to csv         
    if len(chunks) > 0:
        df = pd.concat(chunks, ignore_index=True)
        df.to_csv(f"./data/subreddits/dataframe_{num_dataframes}")

### Loading the data into dataframes

In [12]:
load_zst_json("./reddit/subreddits/subreddits_2025-01.zst")

Amount of subreddits processed: 21865531it [44:46, 8140.26it/s] 


### Custom display options to enable us to see all columns of subreddits_2025_01.zst

In [21]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_rows', 50)

### Display subreddits_2025_01.zst

Function for loading dataframes and cleaning them

In [36]:
def load_dataframe_from_csv(number):
        path = "./data/subreddits"
        df = pd.read_csv(os.path.join(path, f"dataframe_{number}.csv"))
        print(f"Loaded dataframe_{number}.csv with shape: {df.shape}")
        df=df.dropna(subset=['description'])
        df=df[['title','description','display_name','url']]
        print(f'Dataframe_{number}.csv after dropping rows with NaN in description and selecting columns: {df.shape}')

        return df


Now we load all files into one big csv file

In [39]:
dataframes=[]
for i in range(22):
    df = load_dataframe_from_csv(i)
    dataframes.append(df)

big_df = pd.concat(dataframes, ignore_index=True)

C:\Users\elisa\AppData\Local\Temp\ipykernel_31720\91400399.py:3: DtypeWarning: Columns (0: advertiser_category, 1: quarantine_message, 2: quarantine_message_html, 3: quarantine_permissions, 4: is_default_banner, 5: is_default_icon, 6: interstitial_warning_message, 7: content_category) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(path, f"dataframe_{number}.csv"))


Loaded dataframe_0.csv with shape: (1000000, 108)
Dataframe_0.csv after dropping rows with NaN in description and selecting columns: (190244, 4)


C:\Users\elisa\AppData\Local\Temp\ipykernel_31720\91400399.py:3: DtypeWarning: Columns (0: advertiser_category, 1: quarantine_message, 2: quarantine_message_html, 3: quarantine_permissions, 4: interstitial_warning_message, 5: content_category) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(path, f"dataframe_{number}.csv"))


Loaded dataframe_1.csv with shape: (1000000, 106)
Dataframe_1.csv after dropping rows with NaN in description and selecting columns: (183174, 4)


C:\Users\elisa\AppData\Local\Temp\ipykernel_31720\91400399.py:3: DtypeWarning: Columns (0: advertiser_category, 1: banner_img, 2: banner_size, 3: icon_img, 4: icon_size, 5: quarantine_message, 6: quarantine_message_html, 7: quarantine_permissions, 8: interstitial_warning_message, 9: content_category) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(path, f"dataframe_{number}.csv"))


...

Loaded dataframe_19.csv with shape: (1000000, 103)
Dataframe_19.csv after dropping rows with NaN in description and selecting columns: (2, 4)


C:\Users\elisa\AppData\Local\Temp\ipykernel_31720\91400399.py:3: DtypeWarning: Columns (0: description, 1: header_title, 2: is_default_banner, 3: key_color, 4: link_flair_position, 5: submit_link_label, 6: submit_text, 7: submit_text_html, 8: submit_text_label, 9: wiki_enabled) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(path, f"dataframe_{number}.csv"))


Loaded dataframe_20.csv with shape: (1000000, 104)
Dataframe_20.csv after dropping rows with NaN in description and selecting columns: (3, 4)


C:\Users\elisa\AppData\Local\Temp\ipykernel_31720\91400399.py:3: DtypeWarning: Columns (0: advertiser_category, 1: banner_background_color, 2: banner_background_image, 3: community_icon, 4: description, 5: emojis_custom_size, 6: header_img, 7: header_size, 8: header_title, 9: is_crosspostable_subreddit, 10: is_default_banner, 11: is_default_icon, 12: key_color, 13: link_flair_position, 14: mobile_banner_image, 15: primary_color, 16: submit_link_label, 17: submit_text, 18: submit_text_html, 19: submit_text_label, 20: wiki_enabled, 21: interstitial_warning_message, 22: quarantine_message, 23: quarantine_message_html, 24: quarantine_permissions, 25: content_category) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(path, f"dataframe_{number}.csv"))


Loaded dataframe_21.csv with shape: (865531, 109)
Dataframe_21.csv after dropping rows with NaN in description and selecting columns: (74247, 4)


In [47]:
# size of the big dataframe
print(f"Big dataframe shape: {big_df.shape}")

Big dataframe shape: (908790, 4)


In [ ]:
# save the big dataframe to csv
big_df.to_csv("./data/subreddits/big_dataframe.csv", index=False)

In [46]:
big_df.loc[big_df["display_name"] == "leagueoflegends"]['description'].values[0]

'1. **[Patch 25.S1.3 Bug Megathread](https://www.reddit.com/r/leagueoflegends/comments/1ii1zb4/patch_25s13_bug_megathread/)**\n\n\n[](https://discord.gg/lol)\n\n> [](https://lolesports.com/en-GB#promo) \n\n\n\n### Announcements\n[**Patch 25.S1.3 Notes**](https://www.leagueoflegends.com/en-gb/news/game-updates/patch-25-s1-3-notes/)\n\n[**Patch 25.S1.3 Bug Megathread**](https://www.reddit.com/r/leagueoflegends/comments/1ii1zb4/patch_25s13_bug_megathread/)\n\n\n\n\n\n\n\n###Subreddit Information\n\n[**Read subreddit rules**](/r/leagueoflegends/w/subredditrules#rules)\n\n[**Subreddit Discord**](http://discord.gg/lol)\n\n[**Spoiler free Subreddit**](https://goo.gl/F8Zkt6) \n\n###### START AUTO-GENERATION, DO NOT TOUCH\n#### Upcoming Matches\n\nMatch|Date\n:--|--:\n**LEC**|\nPlayoffs - FNC vs. TH|[21h](https://itsalmo.st/match-113476054450558917)\nPlayoffs - G2 vs. GX|[23h](https://itsalmo.st/match-113476054450558921)\n\nMatch|Date\n:--|--:\n**LTA North**|\n\nMatch|Date\n:--|--:\n**LTA South

### Test for subreddits

Load in the SNAP dataset and create set of subreddits to find all subreddits which are in both datasets

In [57]:
SNAP = pd.read_csv('data/redditHyperlinks-column-filtered.csv')
SNAP_subreddits=set(SNAP['SOURCE_SUBREDDIT'].tolist()+SNAP['TARGET_SUBREDDIT'].tolist())
print('SNAP dataset contains', len(SNAP_subreddits), 'unique subreddits')

SNAP dataset contains 35776 unique subreddits


Now we can create a dataframe of all the subreddits in both the SNAP dataset and the subreddits dataset. We check whether matching on the display name or the url is best:

In [76]:
display_df = big_df[big_df['display_name'].isin(SNAP_subreddits)]
print(f"Filtered dataframe on display name shape: {display_df.shape}")
url_df = big_df[big_df['url'].str[3:-1].isin(SNAP_subreddits)]
print(f"Filtered dataframe on url shape: {url_df.shape}")

Filtered dataframe on display name shape: (11117, 4)
Filtered dataframe on url shape: (11119, 4)


We find the subreddits that are not duplicated across these dataframes to check the difference

In [77]:
combined = pd.concat([display_df, url_df], ignore_index=True)
unique_rows = combined.drop_duplicates(keep=False)
unique_rows

,title,description,display_name,url
19152,redditaholes,a place where all the assholes of reddit are s...,RedditAHoles,/r/redditaholes/
20736,testingforme123,for me,Testingforme123,/r/testingforme123/


It is seen that the difference originates from the capital letters in the display name, and it is chosen to use the dataframe which is mathched on the url. This means that we now have 11119 subreddits to work with. 

In [78]:
url_df.to_csv("./data/COMBINED.csv", index=False)